# chat_client.py
**網路、多人遊戲與 Web 全端 › 🚀 延伸:MQTT 捷徑路線(聊天室、多人遊戲、Pico 即時儀表板)**



> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-NETWEB/7/1)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!pip -q install paho-mqtt
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), 'chat_client.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

### MQTT 聊天室(終端機版)— 跟 chat.html 在同一個房間就能互相聊天
MQTT 聊天室(終端機版)— 跟 chat.html 在同一個房間就能互相聊天。對照 DAY5 的 ChatServer.py + ChatClient.py:這裡「沒有伺服器程式」,廣播交給 broker。安裝:pip install paho-mqtt。執行:python chat_client.py 你的暱稱 [房間名稱]。

In [ ]:
# MQTT 聊天室(終端機版)— 跟 chat.html 在同一個房間就能互相聊天

### import json
這一段從「import json」開始,先看懂每一行在做什麼,再按執行看看結果。

In [ ]:
# 對照 DAY5 的 ChatServer.py + ChatClient.py:這裡「沒有伺服器程式」,廣播交給 broker
# 安裝:pip install paho-mqtt
# 執行:python chat_client.py 你的暱稱 [房間名稱]

import json
import sys
import time
import uuid

### ====== 1. 設定 ======
====== 1. 設定 ======。

In [ ]:
import paho.mqtt.client as mqtt

# ====== 1. 設定 ======
NAME   = sys.argv[1] if len(sys.argv) > 1 else input("你的暱稱?") or "Python 同學"
ROOM   = sys.argv[2] if len(sys.argv) > 2 else "demo"
PREFIX = f"pyl-academy/v1/{ROOM}/"
BROKER = "broker.emqx.io"      # 公開測試伺服器(請勿傳個資)
PORT   = 1883                  # Python 直接用 TCP;網頁才需要 WebSocket(8084)

In [ ]:
MY_ID  = "py-" + uuid.uuid4().hex[:8]

### ====== 2. 連上 broker 時:訂閱聊天主題、宣告上線 ======
====== 2. 連上 broker 時:訂閱聊天主題、宣告上線 ======。

### ====== 3. 收到訊息:broker 會把同主題的訊息推過來 ======
====== 3. 收到訊息:broker 會把同主題的訊息推過來 ======。

In [ ]:
# ====== 2. 連上 broker 時:訂閱聊天主題、宣告上線 ======
def 連上了(client, userdata, flags, reason_code, properties):
    print(f"已連上 {BROKER},房間「{ROOM}」。輸入訊息按 Enter 送出,輸入 /quit 離開。")
    client.subscribe(PREFIX + "chat")
    client.publish(PREFIX + "online/" + MY_ID, NAME, retain=True)


# ====== 3. 收到訊息:broker 會把同主題的訊息推過來 ======
def 收到訊息(client, userdata, msg):
    m = json.loads(msg.payload.decode("utf-8"))
    if m["name"] != NAME:
        print(f"\r{m['name']}:{m['text']}\n> ", end="", flush=True)

### 遺囑:這支程式當掉或斷網時,broker 會幫我發布「空字串」→ 網頁的在線名單
遺囑:這支程式當掉或斷網時,broker 會幫我發布「空字串」→ 網頁的在線名單自動把我移除。

In [ ]:
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=MY_ID)
client.on_connect = 連上了
client.on_message = 收到訊息
# 遺囑:這支程式當掉或斷網時,broker 會幫我發布「空字串」→ 網頁的在線名單自動把我移除
client.will_set(PREFIX + "online/" + MY_ID, "", retain=True)
client.connect(BROKER, PORT, keepalive=30)

### ====== 4. 主迴圈:讀鍵盤、發布到聊天主題 ======
====== 4. 主迴圈:讀鍵盤、發布到聊天主題 ======。

In [ ]:
client.loop_start()            # 背景執行緒負責收訊息;主執行緒負責 input(),不用自己開 thread

# ====== 4. 主迴圈:讀鍵盤、發布到聊天主題 ======
try:
    while True:
        text = input("> ").strip()
        if text == "/quit":
            break
        if text:
            client.publish(PREFIX + "chat", json.dumps({"name": NAME, "text": text, "ts": int(time.time() * 1000)}, ensure_ascii=False))
finally:
    client.publish(PREFIX + "online/" + MY_ID, "", retain=True)   # 主動下線
    time.sleep(0.3)
    client.loop_stop()
    client.disconnect()